In [ ]:
#loading metacells of rna and atac data

import anndata
from anndata import AnnData
import pandas as pd
import numpy as np

sc_rna = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/SEAcells/SEACell_summarized_RNA.h5ad")  
sc_atac = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/SEAcells/SEACell_summarized_ATAC.h5ad")
rna_mapping = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/SEAcells/RNA_SEACell_mapping.csv", sep='\t', index_col=0)

gene_peaks_10kb = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Rmd_n_notebook/gene_peak_assignments_10kb.csv")
gene_peaks_20kb = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Rmd_n_notebook/gene_peak_assignments_20kb.csv")
gene_peaks_50kb = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Rmd_n_notebook/gene_peak_assignments_50kb.csv")
gene_peaks_100kb = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Rmd_n_notebook/gene_peak_assignments_100kb.csv")

In [ ]:
#function for normalizing the aggregated data by total counts per SEACell to get relative accessibility/expression values (compositional normalization)
def compositional_normalize(df):
    row_sums = df.sum(axis=1)
    return df.div(row_sums, axis=0)

sc_rna_norm = compositional_normalize(sc_rna)
sc_atac_norm = compositional_normalize(sc_atac)

In [ ]:
#Reindex the dataframes to ensure they are in the same order
sc_atac_norm = sc_atac_norm[sc_rna_norm.obs_names, :]
sc_rna_norm = sc_rna_norm[sc_atac_norm.obs_names, :]
print(sc_atac_norm.obs_names.equals(sc_rna_norm.obs_names))  # Should return True

sc_rna_norm.shape

In [ ]:
#Substract the genes that are present in the gene_peaks_10kb dataframe from the sc_rna_norm dataframe, to only keep the genes that have peaks assigned to them
genes_with_peaks_10kb = gene_peaks_10kb["gene_id"].tolist()
sc_rna_norm = sc_rna_norm[:, genes_with_peaks_10kb] 
print(sc_rna_norm.shape)

In [ ]:
#taking the minimum non-zero value in the sc_rna_norm matrix to add it to all values before log transformation to avoid taking log of zero
import numpy as np
non_zero_mask = (sc_rna_norm.X > 0)
epsilon_rna = np.min(sc_rna_norm.X[non_zero_mask])

In [ ]:
#taking the minimum non-zero value in the sc_atac_norm matrix to add it to all values before log transformation to avoid taking log of zero
non_zero_mask = (sc_atac_norm.X > 0)
epsilon_at= np.min(sc_atac_norm.X[non_zero_mask])